In [7]:
pip install openpyxl

Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 250 kB 875 kB/s eta 0:00:01
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [10]:
pip install nbformat

Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 78 kB 791 kB/s eta 0:00:01
     |████████████████████████████████| 90 kB 3.4 MB/s eta 0:00:011
     |████████████████████████████████| 354 kB 3.8 MB/s eta 0:00:01
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install plotly


Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 9.9 MB 8.9 MB/s eta 0:00:01
     |████████████████████████████████| 432 kB 10.2 MB/s eta 0:00:01
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [20]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
import plotly.graph_objects as go

# -------------------------------
# Smart File Loader (CSV + Excel + ODS)
# -------------------------------
def load_well_dataframe(file_path):
    file_path = Path(file_path)
    suffix = file_path.suffix.lower()

    if suffix == ".csv":
        df = pd.read_csv(file_path)

    elif suffix in [".xls", ".xlsx", ".xlsm", ".xlsb"]:
        df = pd.read_excel(file_path)

    elif suffix == ".ods":
        df = pd.read_excel(file_path, engine="odf")

    else:
        raise ValueError("Unsupported file format")

    return df


# -------------------------------
# Find required columns
# -------------------------------
def find_column(df, possible_names):
    for col in df.columns:
        if col.strip().lower() in possible_names:
            return col
    return None


# -------------------------------
# Compute trajectory (Minimum Curvature)
# -------------------------------
def compute_well_trajectory(df):
    md_col  = find_column(df, {"measured_depth","md","dept","depth"})
    inc_col = find_column(df, {"inclination","inc","incl"})
    azi_col = find_column(df, {"azimuth","azi","azim"})

    if not (md_col and inc_col and azi_col):
        raise ValueError("Required columns (Measured Depth, Inclination, Azimuth) not found")

    md  = df[md_col].astype(float).values
    inc = np.radians(df[inc_col].astype(float).values)
    azi = np.radians(df[azi_col].astype(float).values)

    n = len(md)

    X = np.zeros(n)
    Y = np.zeros(n)
    Z = np.zeros(n)

    for i in range(1, n):
        dmd = md[i] - md[i-1]

        inc1, inc2 = inc[i-1], inc[i]
        azi1, azi2 = azi[i-1], azi[i]

        dogleg = np.arccos(
            np.cos(inc2-inc1) -
            np.sin(inc1)*np.sin(inc2)*(1-np.cos(azi2-azi1))
        )

        rf = 1 if dogleg == 0 else 2/dogleg * np.tan(dogleg/2)

        dX = dmd/2 * (np.sin(inc1)*np.sin(azi1) + np.sin(inc2)*np.sin(azi2)) * rf
        dY = dmd/2 * (np.sin(inc1)*np.cos(azi1) + np.sin(inc2)*np.cos(azi2)) * rf
        dZ = dmd/2 * (np.cos(inc1) + np.cos(inc2)) * rf

        X[i] = X[i-1] + dX
        Y[i] = Y[i-1] + dY
        Z[i] = Z[i-1] + dZ

    return md, X, Y, Z


# -------------------------------
# Plot and export 3 views in one image
# -------------------------------
def plot_and_export_views(md, X, Y, Z, output_path, output_3d_path):
    
    images = []

    # --- Top View ---
    plt.figure()
    plt.plot(X, Y)
    plt.gca().set_aspect('equal', adjustable='box')
    plt.title("Top View")
    plt.xlabel("X"); plt.ylabel("Y")
    plt.savefig("top_tmp.png", dpi=300, bbox_inches="tight")
    plt.close()
    images.append(Image.open("top_tmp.png"))

    # --- Cross Section (Depth Downwards) ---
    plt.figure()
    plt.plot(X, Z)
    plt.gca().invert_yaxis()   # Depth increases downward
    plt.title("Cross Section View")
    plt.xlabel("X"); plt.ylabel("TVD")
    plt.savefig("cross_tmp.png", dpi=300, bbox_inches="tight")
    plt.close()
    images.append(Image.open("cross_tmp.png"))

    # --- Combine Top + Cross into one image ---
    widths, heights = zip(*(i.size for i in images))
    combined = Image.new("RGB", (sum(widths), max(heights)))

    x_offset = 0
    for img in images:
        combined.paste(img, (x_offset,0))
        x_offset += img.size[0]

    combined.save(output_path)
    print(f"Saved combined 2-view image → {output_path}")

    # --- Static True 3D-looking Plot ---
    from mpl_toolkits.mplot3d import Axes3D
    
    fig = plt.figure()
    ax = fig.add_subplot(111, projection='3d')
    ax.plot(X, Y, -Z)   # -Z so depth goes downward visually
    ax.set_title("3D Well Trajectory")
    ax.set_xlabel("X")
    ax.set_ylabel("Y")
    ax.set_zlabel("TVD")
    
    plt.savefig(output_3d_path, dpi=300, bbox_inches="tight")
    plt.close()
    
    print(f"Saved static 3D image → {output_3d_path}")


# -------------------------------
# Interactive 3D in Jupyter
# -------------------------------
def show_interactive_3d(X, Y, Z):
    import plotly.graph_objects as go
    import plotly.io as pio
    
    # Force browser rendering instead of Jupyter MIME
    pio.renderers.default = "browser"

    fig = go.Figure(data=[go.Scatter3d(
        x=X, y=Y, z=-Z,
        mode='lines'
    )])

    fig.update_layout(
        title="Interactive 3D Well Trajectory",
        scene=dict(
            xaxis_title="X",
            yaxis_title="Y",
            zaxis_title="TVD"
        )
    )

    fig.show()


# -------------------------------
# MASTER FUNCTION
# -------------------------------
def generate_well_plots_from_file(file_path):
    file_path = Path(file_path)
    
    df = load_well_dataframe(file_path)
    md, X, Y, Z = compute_well_trajectory(df)
    
    # Output paths in same folder as input file
    output_2view = file_path.parent / f"{file_path.stem}_well_views.png"
    output_3d   = file_path.parent / f"{file_path.stem}_well_3D.png"
    
    plot_and_export_views(md, X, Y, Z, output_2view, output_3d)
    show_interactive_3d(X, Y, Z)



In [21]:
generate_well_plots_from_file("/Users/apple/Downloads/welldata/2011/SCHOONEBEEK-2601/structured/NLOG_GS_PUB_SCH-2601.xlsx")


Saved combined 2-view image → /Users/apple/Downloads/welldata/2011/SCHOONEBEEK-2601/structured/NLOG_GS_PUB_SCH-2601_well_views.png
Saved static 3D image → /Users/apple/Downloads/welldata/2011/SCHOONEBEEK-2601/structured/NLOG_GS_PUB_SCH-2601_well_3D.png
